# Explore `LibrAI/do-not-answer` for seed prompts

Goal: understand the [Do-Not-Answer](https://huggingface.co/datasets/LibrAI/do-not-answer) train split before adding sampled `question` strings into our ToxSearch seed pool (alongside HarmfulQA / CategoricalHarmfulQA).

This notebook covers:
1. Schema / features
2. Risk / harm taxonomy distributions
3. **Question form:** entries that contain `?` vs declarative sentences (no `?`)
4. Length and uniqueness stats relevant to seeding

In [ ]:
from __future__ import annotations

import re
from collections import Counter

import pandas as pd
from datasets import load_dataset

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 40)

## 1. Load dataset

In [ ]:
ds = load_dataset("LibrAI/do-not-answer", split="train")
print(ds)
print("\nFeatures:")
for name, feat in ds.features.items():
    print(f"  - {name}: {feat}")

In [ ]:
df = ds.to_pandas()
print(f"Rows: {len(df):,}  |  Columns: {len(df.columns)}")
df.head(3)

## 2. Feature overview

Seed-relevant columns are mainly:
- `question` — prompt text we would sample
- `risk_area`, `types_of_harm`, `specific_harms` — taxonomy labels

The remaining columns are model responses / harmfulness / action labels for several LLMs (GPT-4, ChatGPT, Claude, ChatGLM2, Llama-2-7B-Chat, Vicuna-7B). Those are evaluation artifacts, not seed text.

In [ ]:
seed_cols = ["id", "risk_area", "types_of_harm", "specific_harms", "question"]
response_cols = [c for c in df.columns if c not in seed_cols]

print("Seed / taxonomy columns:", seed_cols)
print(f"Response / label columns ({len(response_cols)}):", response_cols)
print("\nNulls in seed columns:")
print(df[seed_cols].isna().sum())
print("\nUnique questions:", df["question"].nunique())
print("Duplicate questions:", len(df) - df["question"].nunique())

In [ ]:
print("risk_area value counts")
display(df["risk_area"].value_counts())

print("\ntypes_of_harm value counts")
display(df["types_of_harm"].value_counts())

print("\nspecific_harms (top 20)")
display(df["specific_harms"].value_counts().head(20))

## 3. Question form: `?` vs sentences

Classification used here:
- **has_question_mark**: text contains at least one `?`
- **sentence_like**: no `?` anywhere (imperatives / statements / requests)

We also report whether the string *ends* with `?` (stricter interrogative ending).

In [ ]:
q = df["question"].astype(str).str.strip()

has_qmark = q.str.contains(r"\?", regex=True)
ends_with_qmark = q.str.endswith("?")
sentence_like = ~has_qmark

summary = pd.DataFrame(
    {
        "count": [
            int(has_qmark.sum()),
            int(sentence_like.sum()),
            int(ends_with_qmark.sum()),
            int((has_qmark & ~ends_with_qmark).sum()),
            len(df),
        ],
    },
    index=[
        "contains '?'",
        "no '?' (sentence-like)",
        "ends with '?'",
        "contains '?' but does not end with '?'",
        "total",
    ],
)
summary["pct"] = (100 * summary["count"] / len(df)).round(2)
summary

In [ ]:
# Opening-word patterns (rough surface form)
openers = q.str.extract(r"^(?P<opener>[A-Za-z']+)", expand=True)["opener"].str.lower()

interrogative_openers = {
    "what", "why", "how", "when", "where", "who", "whom", "which",
    "can", "could", "would", "should", "is", "are", "do", "does", "did",
    "will", "may", "might", "have", "has", "was", "were",
}
imperative_openers = {
    "tell", "describe", "explain", "write", "give", "list", "show",
    "create", "generate", "provide", "make", "say", "talk", "help",
    "act", "pretend", "ignore", "please",
}

form = pd.DataFrame({"question": q, "opener": openers, "has_?": has_qmark, "ends_?": ends_with_qmark})
form["opener_class"] = "other"
form.loc[form["opener"].isin(interrogative_openers), "opener_class"] = "interrogative-ish"
form.loc[form["opener"].isin(imperative_openers), "opener_class"] = "imperative-ish"

print("Opener class × has '?'")
display(pd.crosstab(form["opener_class"], form["has_?"], margins=True))

print("\nTop openers among sentence-like (no '?')")
display(form.loc[sentence_like, "opener"].value_counts().head(15))

print("\nTop openers among entries with '?'")
display(form.loc[has_qmark, "opener"].value_counts().head(15))

### Examples

In [ ]:
print("=== Sample with '?' ===")
display(df.loc[has_qmark, ["id", "risk_area", "types_of_harm", "question"]].sample(min(10, int(has_qmark.sum())), random_state=42))

print("=== Sample sentence-like (no '?') ===")
display(df.loc[sentence_like, ["id", "risk_area", "types_of_harm", "question"]].sample(min(10, int(sentence_like.sum())), random_state=42))

## 4. Length / taxonomy vs question form

In [ ]:
stats = df.assign(
    n_chars=q.str.len(),
    n_words=q.str.split().str.len(),
    form=has_qmark.map({True: "contains '?'", False: "no '?'"}),
)

display(stats.groupby("form")[["n_chars", "n_words"]].describe().round(1))

print("\nForm mix by risk_area")
display(pd.crosstab(stats["risk_area"], stats["form"], normalize="index").mul(100).round(1))

## 5. Seed-sampling sketch (same role as HarmfulQA)

For ToxSearch we only need the `question` text (exported as a `questions` column). Example: stratified sample by `risk_area`.

In [ ]:
N_SAMPLE = 100  # change as needed
SEED = 42

seed_pool = (
    df[["question", "risk_area", "types_of_harm", "specific_harms"]]
    .assign(questions=lambda x: x["question"].astype(str).str.strip())
    .drop_duplicates(subset=["questions"])
    .reset_index(drop=True)
)

# Approximate equal per risk_area, then top up if short
areas = seed_pool["risk_area"].unique().tolist()
per = max(1, N_SAMPLE // max(len(areas), 1))
parts = []
for area in areas:
    g = seed_pool[seed_pool["risk_area"] == area]
    parts.append(g.sample(n=min(per, len(g)), random_state=SEED))
sampled = pd.concat(parts, ignore_index=True)
if len(sampled) < N_SAMPLE:
    remain = seed_pool[~seed_pool["questions"].isin(sampled["questions"])]
    need = N_SAMPLE - len(sampled)
    if need > 0 and len(remain) > 0:
        sampled = pd.concat(
            [sampled, remain.sample(n=min(need, len(remain)), random_state=SEED)],
            ignore_index=True,
        )

print(f"Unique pool: {len(seed_pool)} | sampled: {len(sampled)}")
print(sampled["risk_area"].value_counts())
print("\nSampled form mix:")
print(sampled["questions"].str.contains(r"\?", regex=True).value_counts().rename({True: "has ?", False: "no ?"}))

# Preview seed CSV shape (not written unless you uncomment)
out_preview = sampled[["questions"]].copy()
display(out_preview.head())
# out_preview.to_csv("../../data/do_not_answer_sample.csv", index=False)

## Takeaways

- Train split has **939** rows and a clear `question` column suitable for seed prompts.
- Many prompts are **imperatives / statements without `?`** (e.g. "Tell me…", "Describe…"), not only interrogatives.
- Taxonomy columns (`risk_area`, `types_of_harm`, `specific_harms`) are useful for stratified sampling, similar in spirit to CategoricalHarmfulQA categories.
- For ToxSearch seeding, keep only `questions` text; ignore model response / harmful / action columns.